# YOLO26 본 학습 (Colab)

데이터는 로컬에서 만든 yolo.zip (재원님 라벨 수정 + 시드 410 분할 적용).
ultralytics 버전은 로컬이랑 똑같이 8.4.183.

In [ ]:
# 로컬이랑 같은 버전으로 설치
!pip install -q ultralytics==8.4.183
!pip install -q -U kaggle

import torch
import ultralytics

print("ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

## W&B 로그인

키는 Colab 보안 비밀(WANDB_API_KEY)에서 읽음. 노트북엔 안 남음.
팀: codeit-prj1-team2 / 프로젝트: pill-object-detection

In [ ]:
!pip install -q wandb

import wandb
from google.colab import userdata

wandb.login(key=userdata.get("WANDB_API_KEY"))  # 보안 비밀에서 키 읽기

## 1. 데이터 준비

드라이브의 yolo.zip을 Colab으로 풀어서 씀 (드라이브에서 바로 읽으면 느림).
윈도우 zip이라 경로 구분자 \ → / 로 바꿔서 풀고, data.yaml 경로도 Colab 기준으로 고침.

In [ ]:
# 드라이브 전체에서 yolo.zip 찾기
!find /content/drive/MyDrive -iname "yolo*.zip" 2>/dev/null

# 내 드라이브 바로 아래 목록
!ls /content/drive/MyDrive

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile
import yaml

drive.mount("/content/drive")

ZIP_PATH = Path("/content/drive/MyDrive/pill data/yolo.zip")
YOLO_DIR = Path("/content/yolo")

# 윈도우 zip이라 \ 를 / 로 바꿔가며 하나씩 풀기
with zipfile.ZipFile(ZIP_PATH) as z:
    for info in z.infolist():
        name = info.filename.replace("\\", "/")
        if name.endswith("/"):
            continue  # 폴더 항목은 건너뜀
        out = YOLO_DIR / name
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_bytes(z.read(info))

# data.yaml 경로를 Colab 기준으로
cfg_path = YOLO_DIR / "data.yaml"
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
cfg["path"] = str(YOLO_DIR)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")

for s in ["train", "val"]:
    print(s, "이미지:", len(list((YOLO_DIR / "images" / s).glob("*.png"))),
          "/ 라벨:", len(list((YOLO_DIR / "labels" / s).glob("*.txt"))))
print("클래스 수:", len(cfg["names"]))

In [ ]:
# 새 zip이 맞는지 박스 수로 확인 (수정 반영이면 771)
saved = sum(len(p.read_text().splitlines()) for p in (YOLO_DIR / "labels").rglob("*.txt"))
print("박스:", saved)
assert saved == 771, "예전 zip인 것 같음. 드라이브 yolo.zip 확인 필요"

## 1-2. test 사진 준비

드라이브 test_images.zip에서 풀어서 씀.
(처음엔 Kaggle API로 받았는데 전체 다운로드는 막혀 있고, 하나씩 받으면 요청 제한에 걸려서 30분 넘게 걸림. 그래서 한 번 받은 걸 드라이브에 저장해둠)

In [ ]:
import zipfile

TEST_ZIP = Path("/content/drive/MyDrive/pill data/test_images.zip")
TEST_IMG_DIR = Path("/content/test_images")

with zipfile.ZipFile(TEST_ZIP) as z:
    z.extractall("/content")  # zip 안이 test_images/*.png 라서 /content/test_images로 풀림

print("test 이미지:", len(list(TEST_IMG_DIR.glob("*.png"))))

## 2. 본 학습 (기준 실험, 라벨 수정 반영)

처음 돌린 건 재원님 수정이 yolo.zip에 안 들어가 있었음 (변환 셀이 수정 셀보다 앞에 있었음).
고친 zip(박스 771)으로 다시 돌림. 이름 끝에 _fix 붙여서 예전 run이랑 구분.
seed 0, 1, 2로 점수가 얼마나 흔들리는지도 같이 봄.

In [ ]:
import wandb
from ultralytics import YOLO, settings

settings.update({"wandb": True})

WANDB_ENTITY = "codeit-prj1-team2"
WANDB_PROJECT = "pill-object-detection"

MODEL = "yolo26s"
EPOCHS = 100
RUNS_DIR = Path("/content/drive/MyDrive/pill data/runs")

def train_one(seed, exp="", **aug):
    # exp: 실험 이름 (run 이름 끝에 붙음), aug: 기본값에서 바꿀 설정
    run_name = f"{MODEL}_e{EPOCHS}_s{seed}_fix" + (f"_{exp}" if exp else "")

    # 기본값 먼저, 실험에서 넘긴 값으로 덮어씀
    args = dict(
        data=str(YOLO_DIR / "data.yaml"),
        epochs=EPOCHS,
        patience=30,
        imgsz=640,
        batch=16,
        device=0,
        seed=seed,
        workers=2,
        project=str(RUNS_DIR),
        name=run_name,
        exist_ok=True,
    )
    args.update(aug)

    wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name=f"cy_{run_name}",
        tags=["yolo26", "label-fixed"] + ([exp] if exp else ["baseline"]),
        config={
            "model": MODEL,
            "split_seed": 410,
            "data": "yolo.zip 수정판 (재원 라벨 수정 반영, 박스 771)",
            **{k: v for k, v in args.items() if k not in ("data", "project", "name")},
        },
    )

    model = YOLO(f"{MODEL}.pt")
    model.train(**args)
    return run_name

## 3. test 예측 → 제출 csv

학습한 모델로 바로 예측. csv는 run 폴더(드라이브)에 저장되고 내 컴퓨터로도 받아짐.
처음엔 s0_fix로 로컬(VS Code) 결과 4902행이랑 맞는지 확인.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

RUNS_DIR = Path("/content/drive/MyDrive/pill data/runs")  # 학습 결과 폴더

import re
import pandas as pd
from google.colab import files

COLS = ["image_id", "category_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"]
N_TEST = len(list(TEST_IMG_DIR.glob("*.png")))

def predict_run(run_name, imgsz=640, conf=0.01, download=True):
    run_dir = RUNS_DIR / run_name
    weights = run_dir / "weights" / "best.pt"
    if not weights.exists():
        print(f"[건너뜀] {run_name}: best.pt 없음")
        return

    model = YOLO(weights)
    rows = []
    for r in model.predict(
        source=str(TEST_IMG_DIR),  # 폴더째 넘겨야 한 장씩 읽음
        imgsz=imgsz,               # 학습 때랑 같은 크기
        conf=conf,
        device=0,
        stream=True,
        verbose=False,
    ):
        image_id = int(re.findall(r"\d+", Path(r.path).stem)[-1])
        for (x1, y1, x2, y2), c, s in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
            rows.append([
                image_id,
                int(model.names[int(c)]),  # 0~55 → 원래 약 번호
                round(x1, 2), round(y1, 2),
                round(x2 - x1, 2), round(y2 - y1, 2),
                round(s, 4),
            ])

    sub = pd.DataFrame(rows, columns=COLS)
    sub.insert(0, "annotation_id", range(1, len(sub) + 1))

    sub_path = run_dir / f"submission_{run_name}.csv"  # 이름에 run 넣어서 안 헷갈리게
    sub.to_csv(sub_path, index=False)

    n_img = sub["image_id"].nunique()
    note = "" if n_img == N_TEST else "  ← 예측 안 나온 이미지 있음"
    print(f"[완료] {run_name}: {len(sub)}행, 이미지 {n_img}/{N_TEST}{note}")

    if download:
        files.download(str(sub_path))  # 내 컴퓨터로 받기

## 3-1. Kaggle 제출

Colab에서 바로 제출. 키는 보안 비밀 KAGGLE_JSON.
점수는 채점에 1~2분 걸려서 check_scores()로 따로 봄.

In [ ]:
import json, os, subprocess
from google.colab import userdata

COMP = "ai15-level1-project"

def kaggle_login():
    # 보안 비밀의 kaggle.json을 파일로 (세션마다 새로 필요)
    cred = json.loads(userdata.get("KAGGLE_JSON"))
    d = Path.home() / ".kaggle"
    d.mkdir(exist_ok=True)
    (d / "kaggle.json").write_text(json.dumps(cred))
    os.chmod(d / "kaggle.json", 0o600)

def submit(run_name, msg):
    kaggle_login()
    path = RUNS_DIR / run_name / f"submission_{run_name}.csv"
    subprocess.run(["kaggle", "competitions", "submit", "-c", COMP,
                    "-f", str(path), "-m", f"{run_name} {msg}"], check=True)

def check_scores():
    kaggle_login()
    subprocess.run(["kaggle", "competitions", "submissions", "-c", COMP], check=True)

## 4. val–test 차이 원인 보기

test는 정답이 없어서, 사진마다 자신 있게(score 0.5 이상) 찾은 박스 수를 train 알약 수랑 비교함.
test에서 찾은 개수가 확 적으면 못 알아보는 알약이 있다는 뜻.

In [ ]:
RUN = "yolo26s_e100_s0_fix"
sub = pd.read_csv(RUNS_DIR / RUN / f"submission_{RUN}.csv")

# test: 사진마다 score 0.5 이상 박스 수 (하나도 없는 사진은 0)
test_ids = [int(p.stem) for p in TEST_IMG_DIR.glob("*.png")]
test_cnt = (sub[sub["score"] >= 0.5].groupby("image_id").size()
            .reindex(test_ids, fill_value=0))

# train+val: 사진마다 정답 박스 수 (yolo 라벨 줄 수)
gt_cnt = pd.Series([len(p.read_text().splitlines())
                    for p in (YOLO_DIR / "labels").rglob("*.txt")])

print("사진당 알약 수 (train+val 정답)")
print(gt_cnt.value_counts().sort_index().to_string())
print("\n사진당 자신 있게 찾은 수 (test 예측)")
print(test_cnt.value_counts().sort_index().to_string())

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

conf_sub = sub[sub["score"] >= 0.5]

few = test_cnt[test_cnt <= 2].index.tolist()   # 1~2개만 찾은 사진
many = test_cnt[test_cnt >= 5].index.tolist()  # 5개 찾은 사진

random.seed(0)  # 다시 돌려도 같은 사진 나오게
pick_few = random.sample(few, min(4, len(few)))
pick_many = random.sample(many, min(4, len(many)))
print("2개 이하:", pick_few, "/ 5개:", pick_many)

def show(ids, title):
    fig, axes = plt.subplots(1, len(ids), figsize=(5 * len(ids), 7), squeeze=False)
    for ax, iid in zip(axes[0], ids):
        ax.imshow(Image.open(TEST_IMG_DIR / f"{iid}.png"))
        ax.set_title(f"{iid}.png")
        ax.axis("off")
        for b in conf_sub[conf_sub["image_id"] == iid].itertuples():
            ax.add_patch(patches.Rectangle((b.bbox_x, b.bbox_y), b.bbox_w, b.bbox_h,
                                           fill=False, edgecolor="red", linewidth=2))
            ax.text(b.bbox_x, b.bbox_y - 8, f"{b.category_id} ({b.score:.2f})",
                    color="white", fontsize=9, backgroundcolor="red")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show(pick_few, "2개 이하로 찾은 사진")
show(pick_many, "5개 찾은 사진")

In [ ]:
import itertools
from collections import Counter
import numpy as np
import yaml

# yolo 번호 → 원래 약 번호
names = yaml.safe_load((YOLO_DIR / "data.yaml").read_text(encoding="utf-8"))["names"]
idx2cat = {int(k): int(v) for k, v in names.items()}

def iou(a, b):  # a, b = (x, y, w, h)
    ax2, ay2, bx2, by2 = a[0] + a[2], a[1] + a[3], b[0] + b[2], b[1] + b[3]
    iw = max(0, min(ax2, bx2) - max(a[0], b[0]))
    ih = max(0, min(ay2, by2) - max(a[1], b[1]))
    inter = iw * ih
    return inter / (a[2] * a[3] + b[2] * b[3] - inter)

# 1. 같은 알약에 겹친 박스 찾기 (score 0.5 이상끼리)
rows = []
for iid, g in conf_sub.groupby("image_id"):
    bs = g[["bbox_x", "bbox_y", "bbox_w", "bbox_h", "category_id", "score"]].values
    for i, j in itertools.combinations(range(len(bs)), 2):
        v = iou(bs[i], bs[j])
        if v >= 0.5:
            rows.append([iid, int(bs[i][4]), round(bs[i][5], 2), int(bs[j][4]), round(bs[j][5], 2), round(v, 2)])
dup = pd.DataFrame(rows, columns=["image_id", "약A", "scoreA", "약B", "scoreB", "IoU"])

print("겹친 박스 있는 사진:", dup["image_id"].nunique(), "/", N_TEST)
print("\n[5개 사진에서 겹친 것]")
print(dup[dup["image_id"].isin(pick_many)].to_string(index=False))
print("\n[자주 헷갈리는 약 쌍 Top 10]")
print(dup.apply(lambda r: tuple(sorted((r["약A"], r["약B"]))), axis=1).value_counts().head(10).to_string())

# 2. train 박스 수 vs test 예측 수
train_cnt = Counter()
for p in (YOLO_DIR / "labels" / "train").glob("*.txt"):
    for line in p.read_text().splitlines():
        train_cnt[idx2cat[int(line.split()[0])]] += 1

tbl = pd.DataFrame({
    "train박스": pd.Series(train_cnt),
    "test예측": conf_sub["category_id"].value_counts(),
}).fillna(0).astype(int)
tbl["배율"] = (tbl["test예측"] / tbl["train박스"].replace(0, np.nan)).round(1)  # 분포 같으면 약 4.5

print("\n[그림에서 헷갈린 약들]")
print(tbl.loc[[c for c in [3832, 13395, 3743, 28763, 12778] if c in tbl.index]].to_string())
print("\n[배율 높은 약 Top 10 — 모르는 약을 떠안고 있을 후보]")
print(tbl.sort_values("배율", ascending=False).head(10).to_string())

#### 다시 계산: train+val 전체 라벨 기준

위 표는 train 박스만 셌는데, 28763 같은 약은 val에도 있음. 기대값을 전체 라벨(232장) 기준으로 다시 계산.
test 842장 / 라벨 232장 = 약 3.6배가 정상. 기대대비가 1보다 크면 그만큼 많이 예측한 것.

In [ ]:
import pandas as pd
from google.colab import drive
from pathlib import Path
import yaml
from collections import Counter

drive.mount("/content/drive")
RUNS_DIR = Path("/content/drive/MyDrive/pill data/runs")

# yolo 번호 → 원래 약 번호
names_yaml = yaml.safe_load((YOLO_DIR / "data.yaml").read_text(encoding="utf-8"))["names"]
idx2cat = {int(k): int(v) for k, v in names_yaml.items()}

# 기준 run 제출 파일에서 자신 있는 박스(score 0.5 이상)만
sub = pd.read_csv(next((RUNS_DIR / "yolo26s_e100_s0_fix").glob("submission*.csv")))
conf_sub = sub[sub["score"] >= 0.5]

# train만 센 것, train+val 전체 센 것 따로
train_cnt, all_cnt = Counter(), Counter()
for split in ["train", "val"]:
    for p in (YOLO_DIR / "labels" / split).glob("*.txt"):
        for line in p.read_text().splitlines():
            c = idx2cat[int(line.split()[0])]
            all_cnt[c] += 1
            if split == "train":
                train_cnt[c] += 1

ratio = 842 / 232  # test 사진 수 / 라벨 있는 사진 수
tbl = pd.DataFrame({
    "train박스": pd.Series(train_cnt),
    "전체박스": pd.Series(all_cnt),
    "test예측": conf_sub["category_id"].value_counts(),
}).fillna(0).astype(int)
tbl["기대"] = (tbl["전체박스"] * ratio).round(1)          # 분포가 같다면 나올 개수
tbl["기대대비"] = (tbl["test예측"] / tbl["기대"]).round(1)  # 1이면 정상
print(tbl.sort_values("기대대비", ascending=False).head(10).to_string())

#### 결과: 전체 라벨 기준 배율

| 약 | train | 전체 | test 예측 | 기대 | 기대대비 |
|---|---|---|---|---|---|
| 3743 | 3 | 3 | 97 | 10.9 | 8.9 |
| 16551 | 3 | 3 | 93 | 10.9 | 8.5 |
| 34597 | 3 | 3 | 67 | 10.9 | 6.1 |
| 27777 | 3 | 3 | 66 | 10.9 | 6.1 |
| 31885 | 3 | 3 | 62 | 10.9 | 5.7 |
| 13395 | 3 | 3 | 60 | 10.9 | 5.5 |
| 28763 | 3 | 9 | 153 | 32.7 | 4.7 |
| 12778 | 6 | 6 | 95 | 21.8 | 4.4 |
| 2483 | 9 | 9 | 137 | 32.7 | 4.2 |
| 27926 | 3 | 3 | 44 | 10.9 | 4.0 |

- 28763은 val에도 6박스 있어서 위 표(51배)보다 훨씬 낮아짐 (4.7배)
- 그래도 상위 10개 중 7개가 전체 3박스뿐인 약. 3박스 다 train에 있음
- 결론은 같음: 모르는 약을 희귀 약으로 몰아 찍는 걸로 보임

## 5. 실험: 상하 뒤집기

기본은 좌우 뒤집기만 켜져 있음. 상하도 켜서 알약 방향을 4가지로.
회전은 안 넣음 (네모 박스라 돌리면 박스가 커져서 정밀 채점에 불리).
기준: s0_fix Kaggle 0.410, 시드 흔들림 0.02 → 0.43 넘어야 효과 있다고 봄.

In [ ]:
run_name = train_one(0, exp="flipud", flipud=0.5)  # 상하 뒤집기 50%
predict_run(run_name)

In [ ]:
submit("yolo26s_e100_s0_fix_flipud", "상하 뒤집기")

In [ ]:
check_scores()

### 결과: 상하 뒤집기

| run | val mAP50-95 | Kaggle |
|---|---|---|
| s0_fix / s1_fix / s2_fix (기준) | 0.975 / 0.963 / 0.956 | 0.410 / 0.410 / 0.390 |
| s0_fix_flipud | 0.968 | 0.403 |

- val, Kaggle 둘 다 기준 시드 범위 안 → **효과 없음**
- 문제는 비슷한 알약을 구분 못 하는 건데, 뒤집기는 방향만 다양하게 해줄 뿐이라 구분하는 데는 도움이 안 된 걸로 보임
- 다음 실험부터는 상하 뒤집기 빼고 기본값으로 감
- 이번부터 Colab에서 바로 제출함 (submit 함수)

## 6. 실험: 해상도 1280

640은 원본(976×1280)을 절반으로 줄인 것. 헷갈린 약들이 각인 글씨로 구분되는 경우가 많아서 원본 크기로 봄.
1280에서도 안 오르면 해상도는 원인이 아니라고 보고 960은 안 함.
메모리 때문에 batch 16 → 8. 예측도 1280으로 해야 함.
기준: Kaggle 0.43 이상이면 효과 있음.

In [ ]:
run_name = train_one(0, exp="img1280", imgsz=1280, batch=8)
predict_run(run_name, imgsz=1280, download=False)  # 학습 크기랑 맞춤
submit(run_name, "해상도 1280")

In [ ]:
check_scores()

### 결과: 해상도 1280

| run | val mAP50-95 | Kaggle |
|---|---|---|
| 기준 seed 0/1/2 | 0.975 / 0.963 / 0.956 | 0.410 / 0.410 / 0.390 |
| s0_fix_flipud | 0.968 | 0.403 |
| s0_fix_img1280 | 0.965 | 0.403 |

- 원본 크기까지 올려도 그대로 → 해상도는 원인 아님. 960은 안 함
- 증강, 해상도 둘 다 안 움직임 → 학습 방법보다 데이터 한계 (희귀 약 train 3장, test엔 훨씬 많이 나옴)
- 지금 데이터로는 0.40 근처가 천장으로 보임. 실험은 여기서 멈추고 외부 데이터 붙일 준비부터

## 7. 1차 실험 정리 (18시 공유용)

### 7-1. 실험 설정 표
run 폴더마다 있는 args.yaml을 모아서 표로 만듦.
run끼리 값이 다른 설정은 앞쪽으로 빼서 바로 보이게.

In [ ]:
from google.colab import drive
from pathlib import Path
import yaml
import pandas as pd

drive.mount("/content/drive")
RUNS_DIR = Path("/content/drive/MyDrive/pill data/runs")

# run마다 args.yaml(학습 설정 전부) 읽기
rows = {}
for p in sorted(RUNS_DIR.glob("*/args.yaml")):
    rows[p.parent.name] = yaml.safe_load(p.read_text())

cfg = pd.DataFrame(rows).T  # 행: run, 열: 설정
print("run 수:", len(cfg))
print(cfg.index.tolist())

# run끼리 값이 다른 설정만 골라서 보기
diff_cols = [c for c in cfg.columns if cfg[c].astype(str).nunique() > 1]
print("\n다른 설정:", diff_cols)
print(cfg[diff_cols].to_string())

# amp 켜져 있었는지 확인
print("\namp:", cfg["amp"].unique())

# 다른 설정을 앞에 두고 csv로 저장
cfg = cfg[diff_cols + [c for c in cfg.columns if c not in diff_cols]]
cfg.to_csv(RUNS_DIR / "1차_실험설정.csv", encoding="utf-8-sig")
print("\n저장:", RUNS_DIR / "1차_실험설정.csv")

### 7-2. 학습 곡선 (loss, mAP)

run마다 results.csv에 에폭별 loss랑 val 점수가 저장돼 있음.
- 실제로 몇 에폭까지 돌았는지 (patience 30이라 일찍 멈췄을 수 있음)
- val mAP가 가장 높았던 에폭
- train loss는 계속 내려가는데 val loss가 올라가면 과적합
그래프 글자는 한글이 깨져서 영어로 씀.

In [ ]:


import matplotlib.pyplot as plt

# Kaggle 점수 (제출 기록에서 옮김)
KAGGLE = {
    "yolo26s_e100_s0": 0.392,
    "yolo26s_e100_s0_fix": 0.410,
    "yolo26s_e100_s1_fix": 0.410,
    "yolo26s_e100_s2_fix": 0.390,
    "yolo26s_e100_s0_fix_flipud": 0.403,
    "yolo26s_e100_s0_fix_img1280": 0.403,
    "yolo26s_e20_s0_fix_epoch": 0.325,
    "yolo26s_e30_s0_fix_epoch": 0.380,
    "yolo26s_e50_s0_fix_epoch": 0.404,
}

def load_results(run):
    df = pd.read_csv(RUNS_DIR / run / "results.csv")
    df.columns = df.columns.str.strip()  # 열 이름 앞뒤 공백 제거
    return df

MAP = "metrics/mAP50-95(B)"

# 1. run별 요약: 돌린 에폭, val 최고 에폭
summ = []
for run in cfg.index:
    df = load_results(run)
    best = df.loc[df[MAP].idxmax()]  # val mAP50-95 가장 높은 줄
    summ.append({
        "run": run,
        "돌린 에폭": int(df["epoch"].max()),
        "best 에폭": int(best["epoch"]),
        "best val mAP50-95": round(best[MAP], 3),
        "Kaggle": KAGGLE.get(run),
    })
summ = pd.DataFrame(summ).set_index("run")
print(summ.to_string())
summ.to_csv(RUNS_DIR / "1차_결과요약.csv", encoding="utf-8-sig")

# 2. s0_fix 기준으로 train/val loss 비교
df = load_results("yolo26s_e100_s0_fix")
loss_types = [c.split("/")[1] for c in df.columns if c.startswith("train/") and "loss" in c]
print("\nloss 종류:", loss_types)

fig, axes = plt.subplots(1, len(loss_types), figsize=(5 * len(loss_types), 4))
for ax, lt in zip(axes, loss_types):
    ax.plot(df["epoch"], df[f"train/{lt}"], label="train")
    ax.plot(df["epoch"], df[f"val/{lt}"], label="val")
    ax.set_title(f"s0_fix {lt}")
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.show()

# 3. 모든 run의 val mAP50-95 곡선 겹쳐 보기
plt.figure(figsize=(10, 5))
for run in cfg.index:
    d = load_results(run)
    plt.plot(d["epoch"], d[MAP], label=run.replace("yolo26s_e100_", ""))
plt.title("val mAP50-95 by epoch")
plt.xlabel("epoch")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 7-3. confusion matrix, 박스 오류 vs 클래스 오류

s0_fix의 best.pt로 val을 다시 돌려서 confusion matrix를 숫자로 받음.
- 맞춤: 박스도 클래스도 맞음
- 클래스 틀림: 박스 위치는 맞는데 다른 약으로 찍음
- 못 찾음: 정답 알약에 박스 자체가 없음
- 없는 걸 찾음: 알약 없는 곳에 박스
val은 쉬운 시험이라 test 헷갈림(4번 섹션)이랑 같이 봐야 함.

In [ ]:
import numpy as np
from IPython.display import Image, display
from ultralytics import YOLO

RUN = "yolo26s_e100_s0_fix"
model = YOLO(RUNS_DIR / RUN / "weights" / "best.pt")

# val 다시 돌리기 (그림은 run 폴더 안 val_cm에 저장)
metrics = model.val(
    data=str(YOLO_DIR / "data.yaml"), split="val",
    imgsz=640, batch=16, device=0, plots=True,
    project=str(RUNS_DIR / RUN), name="val_cm", exist_ok=True,
)

cm = metrics.confusion_matrix.matrix  # 행: 예측, 열: 정답, 마지막 줄/칸은 배경
nc = cm.shape[0] - 1

correct = np.trace(cm[:nc, :nc])                 # 대각선 = 맞춤
wrong_cls = cm[:nc, :nc].sum() - correct         # 박스는 맞는데 클래스 틀림
missed = cm[nc, :nc].sum()                       # 정답인데 배경으로 = 못 찾음
false_pos = cm[:nc, nc].sum()                    # 배경인데 약으로 = 없는 걸 찾음
total_gt = cm[:, :nc].sum()                      # val 정답 박스 수

print(f"val 정답 박스: {int(total_gt)}")
print(f"맞춤: {int(correct)}")
print(f"클래스 틀림: {int(wrong_cls)}")
print(f"못 찾음: {int(missed)}")
print(f"없는 걸 찾음: {int(false_pos)}")

# 헷갈린 약 쌍 (정답 → 예측)
names = model.names  # 0~55 → 원래 약 번호
pairs = [(names[g], names[p], int(cm[p, g]))
         for p in range(nc) for g in range(nc) if p != g and cm[p, g] > 0]
print("\n[val에서 헷갈린 약]")
print(pd.DataFrame(pairs, columns=["정답", "예측", "개수"])
      .sort_values("개수", ascending=False).to_string(index=False))

# 정규화된 confusion matrix 그림
display(Image(str(RUNS_DIR / RUN / "val_cm" / "confusion_matrix_normalized.png"), width=900))

#### 결과: ultralytics confusion matrix

| 구분 | 개수 |
|---|---|
| val 정답 박스 | 159 |
| 맞춤 | 50 |
| 클래스 틀림 | 109 |
| 못 찾음 | 0 |
| 없는 걸 찾음 | 1717 |

- 같은 val에서 클래스별 mAP50은 거의 다 0.995인데, 여기선 159개 중 50개만 맞췄다고 나옴 → 서로 안 맞음
- 그림도 대각선이 비어 있고 진한 칸이 밖에 흩어져 있음
- 행/열을 반대로 읽어도 대각선 합(50)은 같아서 읽는 방향 문제는 아님
- 8.4.183 + YOLO26 조합에서 confusion matrix 계산이 제대로 안 되는 걸로 보임. 이 숫자랑 그림은 안 씀
- 그래서 아래에서 직접 셈

In [ ]:
# ultralytics confusion matrix가 mAP랑 안 맞아서 직접 셈
def iou_xyxy(a, b):
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    area = lambda r: (r[2] - r[0]) * (r[3] - r[1])
    return inter / (area(a) + area(b) - inter)

rows, fp = [], 0
for r in model.predict(source=str(YOLO_DIR / "images" / "val"), imgsz=640,
                       conf=0.25, device=0, stream=True, verbose=False):
    h, w = r.orig_shape
    lbl = YOLO_DIR / "labels" / "val" / f"{Path(r.path).stem}.txt"

    # 정답: yolo 형식(가운데 x, y, 너비, 높이, 0~1) → 픽셀 좌표
    gts = []
    for line in lbl.read_text().splitlines():
        c, x, y, bw, bh = map(float, line.split())
        gts.append((int(c), [(x - bw/2)*w, (y - bh/2)*h, (x + bw/2)*w, (y + bh/2)*h]))

    preds = list(zip(r.boxes.cls.int().tolist(), r.boxes.xyxy.tolist()))
    used = set()  # 이미 정답이랑 짝지어진 예측

    for gc, gb in gts:
        # 아직 안 쓰인 예측 중 제일 많이 겹치는 것
        best_i, best_iou = -1, 0.0
        for i, (pc, pb) in enumerate(preds):
            if i not in used and iou_xyxy(gb, pb) > best_iou:
                best_i, best_iou = i, iou_xyxy(gb, pb)

        if best_iou >= 0.5:
            used.add(best_i)
            kind = "맞춤" if preds[best_i][0] == gc else "클래스 틀림"
            pred_name = names[preds[best_i][0]]
        else:
            kind = "박스 어긋남" if best_iou > 0.1 else "못 찾음"
            pred_name = None
        rows.append([Path(r.path).stem, names[gc], pred_name, round(best_iou, 2), kind])

    fp += len(preds) - len(used)

res = pd.DataFrame(rows, columns=["사진", "정답", "예측", "IoU", "결과"])
print(res["결과"].value_counts().to_string())
print("없는 걸 찾음:", fp)
print("\n[맞춤 아닌 것]")
print(res[res["결과"] != "맞춤"].to_string(index=False))

#### 결과: 직접 센 박스 오류 vs 클래스 오류 (conf 0.25)

| 구분 | 개수 |
|---|---|
| 맞춤 | 149 / 159 |
| 클래스 틀림 | 6 |
| 박스 어긋남 | 0 |
| 못 찾음 | 4 |
| 없는 걸 찾음 | 1 |

- 149/159(약 94%)로 val mAP 0.97이랑 맞음 → 위 ultralytics 결과가 이상했던 게 맞음
- 박스 어긋남 0개, 클래스 틀린 것도 IoU 0.97~0.99 → 위치는 거의 정확함. **틀리는 건 박스가 아니라 클래스**
- 클래스 틀림 6개는 전부 정답 20238 → 28763 또는 22347로 찍음
- 못 찾음 4개 중 3개가 30308 (클래스별 표에서도 mAP 0.495로 제일 낮았던 약)
- 28763, 22347은 4번 test 분석에서 과하게 예측된 약, 자주 헷갈린 쌍으로 나왔던 약 → val이랑 test에서 같은 약들이 문제
- 같은 약 조합을 각도(70, 75, 90)만 바꾼 사진에서 같은 실수가 반복됨 → 실수 10개라기보다 조합 3~4개의 실수

In [ ]:
# 파일 이름 맨 앞 부분(첫 _ 앞)이 약 조합
combo = lambda p: p.stem.split("_")[0]

train_combo = {combo(p) for p in (YOLO_DIR / "images" / "train").glob("*.png")}
val_imgs = list((YOLO_DIR / "images" / "val").glob("*.png"))
val_combo = {combo(p) for p in val_imgs}

shared = val_combo & train_combo  # train에도 있는 val 조합
print("train 조합 수:", len(train_combo))
print("val 조합 수:", len(val_combo))
print("train에도 있는 val 조합:", len(shared), "/", len(val_combo))
print("그 조합에 속한 val 사진:", sum(combo(p) in shared for p in val_imgs), "/", len(val_imgs))

#### 결과: train/val 약 조합 겹침

| 구분 | 개수 |
|---|---|
| train 조합 | 89 |
| val 조합 | 25 |
| train에도 있는 val 조합 | 0 / 25 |

- 같은 조합이 train, val에 나눠 들어간 건 없음 → 분할 누수 아님
- val은 처음 보는 조합이지만 약 하나하나는 다 train에서 본 약이라 쉬움
- test엔 학습에 없는 약이 섞여 있음 (4번 분석) → val 0.97, test 0.40 차이는 이걸로 설명됨
- 과적합(loss 곡선, 에폭 실험), 분할 누수(여기) 둘 다 아니니 남는 원인은 test의 모르는 약 → 외부 데이터 필요

#### 확인: val에 나온 약이 train에 다 있는지

분할 원칙상 희귀 약은 train에 고정해서 val 약은 전부 train에 있어야 함. 라벨 파일로 직접 확인.

In [ ]:
# 라벨 파일들에서 클래스 번호만 모으기
def classes_in(split):
    s = set()
    for p in (YOLO_DIR / "labels" / split).glob("*.txt"):
        for line in p.read_text().splitlines():
            s.add(int(line.split()[0]))
    return s

train_cls = classes_in("train")
val_cls = classes_in("val")
only_val = val_cls - train_cls  # val에만 있는 약

print("train 약 종류:", len(train_cls))
print("val 약 종류:", len(val_cls))
print("val에만 있는 약:", [names[c] for c in only_val] if only_val else "없음")
print("val에 없는 약(train에만 있음):", len(train_cls - val_cls))

#### 결과: val 약 종류

| 구분 | 개수 |
|---|---|
| train 약 종류 | 56 |
| val 약 종류 | 30 |
| val에만 있는 약 | 0 |
| val에 안 나오는 약 | 26 |

- val에 나온 약은 전부 train에 있음 → val은 아는 약만 나오는 시험, 그래서 쉬움
- 56종 중 26종은 val에 아예 없음 → val 0.97은 30종에 대한 점수. 나머지 26종은 val로 평가가 안 됨
- 2차에서 다시 나눌 때 val에 약 종류가 최대한 많이 들어가게 하는 게 좋아 보임

## 8. 실험: 에폭 수

학습 곡선 보니까 val mAP가 20~30에폭에서 거의 다 올라옴. 100에폭은 근거 없이 정한 거라 다시 찾아봄.
seed 0, 수정판 데이터, 나머지는 기준 실험이랑 같고 에폭만 20 / 30 / 50으로.
patience 30은 그대로라 20, 30에폭은 사실상 조기 종료 없음.
주의: 처음 3에폭 워밍업, 마지막 10에폭 모자이크 끔도 같이 따라 바뀜.

In [ ]:
# train_one이 전역 EPOCHS로 run 이름이랑 에폭을 정해서, 돌기 전에 바꿔줌
for ep in [20, 30, 50]:
    EPOCHS = ep
    run_name = train_one(0, exp="epoch")       # 예: yolo26s_e20_s0_fix_epoch
    predict_run(run_name, download=False)      # csv는 드라이브 run 폴더에 저장
    submit(run_name, f"에폭 {ep}")

### 결과: 에폭 수

| run | best 에폭 | best val | Kaggle |
|---|---|---|---|
| e20 | 20 | 0.858 | 0.325 |
| e30 | 27 | 0.949 | 0.380 |
| e50 | 49 | 0.974 | 0.404 |
| e100 (s0_fix) | 96 | 0.976 | 0.410 |

- val이랑 Kaggle 순서가 같음 (20 < 30 < 50 ≈ 100). 에폭 고르는 데는 val 써도 됨
- 20, 30은 best가 거의 마지막 에폭 → 끝까지 오르는 중이었음, 덜 학습됨
- 50이랑 100은 val, Kaggle 둘 다 시드 흔들림(0.02) 안 → 1차 최적은 50 근처. 시간은 절반
- 100에폭에서도 안 떨어짐 → 과적합 아님
- 100에폭 run의 30에폭 시점이랑 30에폭 run은 다름. 짧게 돌리면 워밍업, 모자이크 끔, 학습률 감소가 다 같이 압축됨
- seed 0 한 번씩이라 경향 정도로만 봄